<a href="https://colab.research.google.com/github/Raymondycp/QuantFinance_AlgoTradingStrategy/blob/main/%5BQuantFinance%5DPositionStrategy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Best posistion Kelly formula
#### Winrate be 50%, after 1000 times, What is the total amount?
#### let winrate be 50%, Find maxium of y and x
#### y = 100*(1+2x)^n*(1-x)^n
#### y = 100*(1+2x)*(1-x)

In [ ]:
pip install dash plotly numpy

In [ ]:
import numpy as np
from dash import Dash, dcc, html, Input, Output, State
import plotly.graph_objs as go

# Core simulation logic
def simulate_trades(bet_pct, win_prob, payout, num_trades, use_seed, starting_capital, leverage):
    if use_seed:
        np.random.seed(42)
    else:
        np.random.seed(None)

    capital = [starting_capital]
    outcomes = np.random.choice([1, 0], size=num_trades, p=[win_prob, 1 - win_prob])
    for outcome in outcomes:
        current = capital[-1]
        base_bet = current * (bet_pct / 100)
        leveraged_bet = base_bet * leverage

        if outcome == 1:
            current += leveraged_bet * payout
        else:
            current -= leveraged_bet

        capital.append(current)
    return capital

# Calculate key metrics
def calculate_stats(curve):
    peak = float(curve[0])
    max_drawdown = 0.0
    for x in curve:
        x = float(x)
        if x > peak:
            peak = x
        dd = (peak - x) / peak
        max_drawdown = max(max_drawdown, dd)
    return {
        'final': round(float(curve[-1]), 2),
        'max_drawdown': round(float(max_drawdown) * 100, 2)
    }

# Initialize Dash App
app = Dash(__name__)
server = app.server

app.layout = html.Div([
    html.H2("Trading Simulator: Position Sizing & Leverage Impact", style={'textAlign': 'center'}),

    html.Div([
        html.Label("Win Rate (%)"),
        dcc.Slider(id='win-prob', min=10, max=90, step=1, value=50,
                   marks={i: f"{i}%" for i in range(10, 100, 10)}),
    ], style={'marginBottom': 20}),

    html.Div([
        html.Label("Payout Multiplier (Win Gain)"),
        dcc.Slider(id='payout', min=0.5, max=3.0, step=0.1, value=1.0,
                   marks={round(float(i), 1): f"{i:.1f}" for i in np.arange(0.5, 3.1, 0.5)})
    ], style={'marginBottom': 20}),

    html.Div([
        html.Label("Leverage (e.g., 1 = none, 2 = 2x exposure)"),
        dcc.Slider(id='leverage', min=1, max=10, step=0.5, value=1,
                   marks={i: f"{i}x" for i in range(1, 11)}),
    ], style={'marginBottom': 20}),

    html.Div([
        html.Label("Number of Trades"),
        dcc.Input(id='num-trades', type='number', value=100, min=10, step=10),
    ], style={'marginBottom': 20}),

    html.Div([
        html.Label("Starting Capital"),
        dcc.Input(id='start-capital', type='number', value=1.0, min=0.01, step=0.01),
    ], style={'marginBottom': 20}),

    html.Div([
        html.Label("Bet Sizes (%) (comma-separated, e.g. 1,5,10,20,50)"),
        dcc.Input(id='bet-sizes', type='text', value="1,5,10,25,50,100", style={'width': '100%'}),
    ], style={'marginBottom': 20}),

    dcc.Checklist(
        id='use-seed',
        options=[{'label': 'Use Fixed Random Seed (repeatable)', 'value': 'fixed'}],
        value=['fixed'],
        style={'marginBottom': 30}
    ),

    html.Button("Run Simulation", id='run-btn', n_clicks=0, style={'marginBottom': 30}),
    dcc.Graph(id='capital-graph'),
    html.Pre(id='summary', style={'fontFamily': 'monospace', 'whiteSpace': 'pre'})
])

@app.callback(
    Output('capital-graph', 'figure'),
    Output('summary', 'children'),
    Input('run-btn', 'n_clicks'),
    State('win-prob', 'value'),
    State('payout', 'value'),
    State('leverage', 'value'),
    State('num-trades', 'value'),
    State('start-capital', 'value'),
    State('bet-sizes', 'value'),
    State('use-seed', 'value'),
)
def update_output(n_clicks, win_prob_percent, payout, leverage, num_trades, start_capital, bet_sizes_str, seed_check):
    win_prob = win_prob_percent / 100
    use_seed = 'fixed' in seed_check
    try:
        bet_sizes = [float(x.strip()) for x in bet_sizes_str.split(',') if x.strip()]
    except ValueError:
        return {}, "⚠️ Error: Invalid input in bet sizes. Use comma-separated numbers (e.g., 1,5,10)"

    traces = []
    summary = []

    for bet_pct in bet_sizes:
        curve = simulate_trades(bet_pct, win_prob, payout, num_trades, use_seed, start_capital, leverage)
        stats = calculate_stats(curve)

        traces.append(go.Scatter(
            x=list(range(len(curve))),
            y=curve,
            mode='lines',
            name=f"{bet_pct}% Bet",
            hoverinfo='name+y'
        ))

        summary.append(
            f"Bet {bet_pct:>5.1f}% | Leverage {leverage}x → Final: {stats['final']:<10} | Max Drawdown: {stats['max_drawdown']}%"
        )

    fig = go.Figure(data=traces)
    fig.update_layout(
        title="Capital Curves for Each Strategy",
        xaxis_title="Trade #",
        yaxis_title="Capital",
        hovermode="x unified"
    )

    return fig, "\n".join(summary)



In [ ]:
if __name__ == '__main__':
    app.run(debug=True)



<IPython.core.display.Javascript object>